# I'm feeling a little Bowie

## Introdução

Digamos que eu esteja me sentindo de uma maneira muito específica, e eu queira ouvir uma música que consoe meu estado de espírito. Para isso, podemos usar técnicas de Processamento de Linguagem Natural (PLN) para analisar o sentimento de uma frase ou texto e, em seguida, recomendar uma música que corresponda a esse sentimento.

Lógicamente, o universo musical é vasto e diverso, há todavia um artista com tamanho alcance e talento que faz com que todos os outros sejam irrelevantes: David Bowie.

Dessa forma, podemos criar um chatbot que, dada uma descrição do seu estado de espírito, responda a música ideal para enriquecer o seu ser.

## Dados

### Obtenção

Em primeiro lugar é necessário obter todas as letras das músicas de David Bowie. Para tanto, foi utilizado o site [Bowie Wonderworld](https://www.bowiewonderworld.com/songs/dblyrics.htm), que contém todas as letras das músicas do artista. A partir desse site, é possível realizar uma cópia bruta (é necessário desativar os scripts do site para tanto) de todas as letras para um arquivo de texto.

### Transformação e anotação

Primeiramente é necessário transformar o arquivo bruto em um arquivo estruturado, de forma que cada música seja representada por uma linha, contendo o título da música, a letra e o sentimento.

Devido ao altíssimo volume de músicas, a anotação manual de sentimentos para cada música seria inviável.
Por tanto, conclui-se que a a utilização de assistência de IA seria a solução mais apropriada.

A transformação e anotação ocorreram por tanto através do Copilot,
 utilizado o modelo GPT-5.6 Sol, janela de contexto de e raciocínio padrão com o seguinte prompt:


```
I need you to create a csv with the title of each song, the lyrics of each song, and a sentence describing the feelings of the song
```


### Aprimoração

O resultado das anotações iniciais todavia não era satisfatório, sentimentos genéricos e muitas vezes repetitivos foram atribuídos a músicas com sentimentos distintos.

As estratégias adotadas para aprimorar as anotações foram as seguintes:

1. Configurar a janela de contexto para 1.1M tokens
2. Aumentar o nível de raciocínio para "xhigh"
3. Descrever melhor o prompt, incluindo exemplos de sentimentos para músicas específicas. O prompt final utilizado foi o seguinte:

```
I need the feeling field to be more descriptive and individual to each song, for instance, Word on a Wing is a song that evokes a feeling of resignation to a higher power, a plea with God for direction, you can look the net for explanations too
```


## Pré-processamento

### Importando dados

In [55]:
import pandas as pd

lyrics = pd.read_csv("bowie_songs.csv")
lyrics

,title,lyrics,feelings
0,1917,(Instrumental),"Wordless and murky, this piece leans on its Fi..."
1,1984,"Someday they won't let you, now you must agree...","A funk-driven warning shot, it wraps claustrop..."
2,1984/Dodo,"Someday they won't let you, but now you must a...","Stitching dystopian alarm to hushed gossip, th..."
3,5:15 The Angels Have Gone,"5:15\nI'm changing trains, this little town\nL...","Set on a rainy platform, this is a farewell we..."
4,'87 And Cry,It's just a one dollar secret\nA lover's secre...,"Frustration curdles into bitterness, a snarlin..."
...,...,...,...
592,Word On A Wing,"In this age of grand delusion, you walked into...","Performed live, the hymn sounds even more expo..."
593,Yassassin,CHORUS\n Yassassin - I'm not a moody guy\n Y...,"A migrant's weary plea for peace, pride and ex..."
594,You Better Tell Her,NaN,"Impatient counsel drives the phrase, somebody ..."
595,You Can't Sit Down,Hey pretty baby! (you can't sit down)\nA don't...,"Irresistible compulsion to move, the beat trea..."


### Limpando dados

In [56]:
lyrics_clean = lyrics.dropna()
lyrics_clean

,title,lyrics,feelings
0,1917,(Instrumental),"Wordless and murky, this piece leans on its Fi..."
1,1984,"Someday they won't let you, now you must agree...","A funk-driven warning shot, it wraps claustrop..."
2,1984/Dodo,"Someday they won't let you, but now you must a...","Stitching dystopian alarm to hushed gossip, th..."
3,5:15 The Angels Have Gone,"5:15\nI'm changing trains, this little town\nL...","Set on a rainy platform, this is a farewell we..."
4,'87 And Cry,It's just a one dollar secret\nA lover's secre...,"Frustration curdles into bitterness, a snarlin..."
...,...,...,...
591,Without You I'm Nothing,Strange infatuation seems to grace the evening...,"Sultry self-abasement, decadent images sliding..."
592,Word On A Wing,"In this age of grand delusion, you walked into...","Performed live, the hymn sounds even more expo..."
593,Yassassin,CHORUS\n Yassassin - I'm not a moody guy\n Y...,"A migrant's weary plea for peace, pride and ex..."
595,You Can't Sit Down,Hey pretty baby! (you can't sit down)\nA don't...,"Irresistible compulsion to move, the beat trea..."


In [4]:
from sklearn.metrics.pairwise import cosine_similarity

def most_similars(my_feeling_vec, vectors, top_n=5):
    similarities = cosine_similarity(my_feeling_vec, vectors)
    most_similar_indices = similarities.argsort()[0][-top_n:][::-1]
    return most_similar_indices

In [39]:
_test = [
    # Se errar esse tem alguma coisa errada
    ("I am on a disordered pilgrimage from occult dread toward desperate romance, a numbed figure pushing himself to feel anything at all.", "Station to Station"),
    # Esse pode ser acertado tanto pela letra quanto pelo sentimento
    ("I feel like I'm cracking under pressure and that only love can save me.", "Under Pressure"),
    # Relativamente fácil também, mas dá para errar
    ("I feel energized and want to dance with my beloved.", "Let's Dance"),
    ("I'm feeling regretful and ashemed for falling so low on my addiction.", "Ashes to Ashes"),
    ("I feel resignation, I just want to understand God's plan for me.", "Word on a Wing"),
    # Esse é o mais difícil, é necessário interpretar o contexto
    ("I'm completely head over heels in love and want to deliver myself completely", "I would be your slave"),
    # Bonus, não tem resposta certa, mas é interessante ver o que o modelo sugere
    ("I'm feeling anxious and stressed about an upcoming event.", "???"),
]

def as_rank(indices):
    return '\n'.join([
        f"\t\t{i + 1}. {lyrics_clean.iloc[indices[i]]['title']}"
        for i in range(5)
    ])

def test(feeling_vectorizer, feeling_vectors, lyrics_vectorizer, lyrics_vectors):
    for phrase, expected in _test:
        most_similar_by_feeling = most_similars(feeling_vectorizer(phrase), feeling_vectors, 5)
        most_similar_by_lyrics = most_similars(lyrics_vectorizer(phrase), lyrics_vectors, 5)
        print(f"Input phrase: {phrase}")
        print(f"\t* Expected: {expected}")
        print(f"\t* Most similar songs by feeling: \n{as_rank(most_similar_by_feeling)}")
        print(f"\t* Most similar songs by lyrics: \n{as_rank(most_similar_by_lyrics)}")
        print("\n")

## TF-IDF

A primeira abordagem, que servirá de base de comparação para as demais, é a utilização de TF-IDF (Term Frequency-Inverse Document Frequency) para transformar os sentimentos das músicas. Essa técnica permite identificar a importância de cada palavra em relação ao conjunto de documentos (neste caso, os sentimentos das músicas).

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer

feelings_tfidf_vectorizer = TfidfVectorizer()
feelings_tfidf_matrix = feelings_tfidf_vectorizer.fit_transform(lyrics_clean['feelings'])

lyrics_tfidf_vectorizer = TfidfVectorizer()
lyrics_tfidf_matrix = lyrics_tfidf_vectorizer.fit_transform(lyrics_clean['lyrics'])

In [42]:
test(
    lambda x: feelings_tfidf_vectorizer.transform([x]),
    feelings_tfidf_matrix,
    lambda x: lyrics_tfidf_vectorizer.transform([x]),
    lyrics_tfidf_matrix
)

Input phrase: I am on a disordered pilgrimage from occult dread toward desperate romance, a numbed figure pushing himself to feel anything at all.
	* Expected: Station to Station
	* Most similar songs by feeling: 
		1. Station To Station
		2. Goodbye Mr. Ed
		3. Soul Love
		4. Hurt
		5. After Today
	* Most similar songs by lyrics: 
		1. I Am With Name
		2. The Pretty Things Are Going To Hell
		3. When The Wind Blows
		4. I Am A Laser
		5. Jewel


Input phrase: I feel energized and want to dance with my beloved.
	* Expected: Let's Dance
	* Most similar songs by feeling: 
		1. Rupert The Riley
		2. Glad I've Got Nobody
		3. Let's Dance - (demo)
		4. Let's Dance
		5. Land Of 1,000 Dances
	* Most similar songs by lyrics: 
		1. Let's Dance - (demo)
		2. Let's Dance
		3. Let's Dance
		4. I Feel So Bad
		5. Magic Dance


Input phrase: I'm feeling regretful for falling so low.
	* Expected: Ashes to Ashes
	* Most similar songs by feeling: 
		1. I Can't Explain
		2. Things To Do
		3. The Secret 

### Word2Vec

In [12]:
import gensim.downloader as api
import numpy as np
import nltk

nltk.download('punkt_tab')
# Load pre-trained Word2Vec model
word2vec_model = api.load("word2vec-google-news-300")
word2vec_model["love"]  # Example of getting the vector for a word

array([ 0.10302734, -0.15234375,  0.02587891,  0.16503906, -0.16503906,
        0.06689453,  0.29296875, -0.26367188, -0.140625  ,  0.20117188,
       -0.02624512, -0.08203125, -0.02770996, -0.04394531, -0.23535156,
        0.16992188,  0.12890625,  0.15722656,  0.00756836, -0.06982422,
       -0.03857422,  0.07958984,  0.22949219, -0.14355469,  0.16796875,
       -0.03515625,  0.05517578,  0.10693359,  0.11181641, -0.16308594,
       -0.11181641,  0.13964844,  0.01556396,  0.12792969,  0.15429688,
        0.07714844,  0.26171875,  0.08642578, -0.02514648,  0.33398438,
        0.18652344, -0.20996094,  0.07080078,  0.02600098, -0.10644531,
       -0.10253906,  0.12304688,  0.04711914,  0.02209473,  0.05834961,
       -0.10986328,  0.14941406, -0.10693359,  0.01556396,  0.08984375,
        0.11230469, -0.04370117, -0.11376953, -0.0037384 , -0.01818848,
        0.24316406,  0.08447266, -0.07080078,  0.18066406,  0.03515625,
       -0.09667969, -0.21972656, -0.00328064, -0.03198242,  0.18

In [15]:
from nltk.tokenize import word_tokenize
from gensim.models import KeyedVectors

def avg_w2v_vec(text, model: KeyedVectors):
    tokens = word_tokenize(text.lower())
    vectors = [
        model[t]
        for t in tokens
        if t in model
    ]

    if not vectors:
        return np.zeros(model.vector_size)

    return np.mean(vectors, axis=0)

[nltk_data] Downloading package punkt_tab to /home/fabio/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [16]:
feelings_word2vec_matrix = [
    avg_w2v_vec(feeling, word2vec_model)
    for feeling in lyrics_clean['feelings']
]

lyrics_word2vec_matrix = [
    avg_w2v_vec(lyric, word2vec_model)
    for lyric in lyrics_clean['lyrics']
]

In [43]:
def vectorize_with_word2vec(text):
    return avg_w2v_vec(text, word2vec_model).reshape(1, -1)

test(
    vectorize_with_word2vec,
    feelings_word2vec_matrix,
    vectorize_with_word2vec,
    lyrics_word2vec_matrix
)

Input phrase: I am on a disordered pilgrimage from occult dread toward desperate romance, a numbed figure pushing himself to feel anything at all.
	* Expected: Station to Station
	* Most similar songs by feeling: 
		1. Station To Station
		2. I Took A Trip On A Gemini Spaceship
		3. When The Wind Blows
		4. Jump They Say
		5. Miracle Goodnight
	* Most similar songs by lyrics: 
		1. The Supermen
		2. Baal's Hymn
		3. Ballad Of The Adventurers
		4. Soul Love
		5. Buddha Of Suburbia


Input phrase: I feel energized and want to dance with my beloved.
	* Expected: Let's Dance
	* Most similar songs by feeling: 
		1. Fun
		2. Foot Stomping / I Wish I Could Shimmy Like My Sister Kate
		3. Knock On Wood
		4. Ragazzo Solo, Ragazza Sola
		5. Miracle Goodnight
	* Most similar songs by lyrics: 
		1. John, I'm Only Dancing (Again)
		2. John, I'm Only Dancing
		3. Sweet Head
		4. Sweet Head - (Haddon Hall rehearsal)
		5. When I Live My Dream


Input phrase: I'm feeling regretful for falling so low.
	

In [21]:
def weighted_avg_w2v_vec(sentence, model, tfidf):
    words = word_tokenize(sentence.lower())
    vectors = []
    weights = []

    for word in words:
        if word in model and word in tfidf:
            vectors.append(model[word])
            weights.append(tfidf[word])

    if not vectors:
        return np.zeros(model.vector_size)

    return np.average(vectors, axis=0, weights=weights)

In [22]:
feeling_weighted_word2vec_matrix = [
    weighted_avg_w2v_vec(feeling, word2vec_model, feelings_tfidf_vectorizer.vocabulary_)
    for feeling in lyrics_clean['feelings']
]

lyrics_weighted_word2vec_matrix = [
    weighted_avg_w2v_vec(lyric, word2vec_model, lyrics_tfidf_vectorizer.vocabulary_)
    for lyric in lyrics_clean['lyrics']
]

In [41]:
def vectorize_with_weighted_word2vec(text):
    return weighted_avg_w2v_vec(text, word2vec_model, feelings_tfidf_vectorizer.vocabulary_).reshape(1, -1)

def vectorize_with_weighted_word2vec_lyrics(text):
    return weighted_avg_w2v_vec(text, word2vec_model, lyrics_tfidf_vectorizer.vocabulary_).reshape(1, -1)

test(
    vectorize_with_weighted_word2vec,
    feeling_weighted_word2vec_matrix,
    vectorize_with_weighted_word2vec_lyrics,
    lyrics_weighted_word2vec_matrix
)


Input phrase: I am on a disordered pilgrimage from occult dread toward desperate romance, a numbed figure pushing himself to feel anything at all.
	* Expected: Station to Station
	* Most similar songs by feeling: 
		1. Station To Station
		2. Aladdin Sane (1913-1938-197?)
		3. The Width Of A Circle
		4. Tryin' To Get To Heaven
		5. China Girl
	* Most similar songs by lyrics: 
		1. Soul Love
		2. God Knows I'm Good
		3. The Drowned Girl
		4. Life On Mars? - (demo)
		5. Soul Love - (demo)


Input phrase: I feel energized and want to dance with my beloved.
	* Expected: Let's Dance
	* Most similar songs by feeling: 
		1. Standing Next To You
		2. Absolute Beginners
		3. Kooks - (demo)
		4. Can You Hear Me
		5. Fun
	* Most similar songs by lyrics: 
		1. I Want My Baby Back - (demo)
		2. I Feel So Bad
		3. When I Live My Dream
		4. I Wanna Be Your Dog
		5. Cygnet Committee


Input phrase: I'm feeling regretful for falling so low.
	* Expected: Ashes to Ashes
	* Most similar songs by feeling: 

## Doc2Vec

In [36]:
from gensim.models.doc2vec import Doc2Vec, TaggedDocument

feelings_tags = [
    TaggedDocument(words=word_tokenize(feeling.lower()), tags=[str(i)])
    for i, feeling in enumerate(lyrics_clean['feelings'])
]

feelings_doc2vec = Doc2Vec(vector_size=100, window=5, min_count=1, workers=4, epochs=40)
feelings_doc2vec.build_vocab(feelings_tags)
feelings_doc2vec.train(feelings_tags, total_examples=feelings_doc2vec.corpus_count, epochs=feelings_doc2vec.epochs)

lyrics_tags = [
    TaggedDocument(words=word_tokenize(lyric.lower()), tags=[str(i)])
    for i, lyric in enumerate(lyrics_clean['lyrics'])
]

lyrics_doc2vec = Doc2Vec(vector_size=100, window=5, min_count=1, workers=4, epochs=40)
lyrics_doc2vec.build_vocab(lyrics_tags)
lyrics_doc2vec.train(lyrics_tags, total_examples=lyrics_doc2vec.corpus_count, epochs=lyrics_doc2vec.epochs)

In [40]:
def as_indices(rank):
    return [int(tag) for tag, _ in rank]

for phrase, expected in _test:
    most_similar_by_feeling = feelings_doc2vec.dv.most_similar(
        [feelings_doc2vec.infer_vector(word_tokenize(phrase.lower()))],
        topn=5
    )
    most_similar_by_lyrics = lyrics_doc2vec.dv.most_similar(
        [lyrics_doc2vec.infer_vector(word_tokenize(phrase.lower()))],
        topn=5
    )
    print(f"Input phrase: {phrase}")
    print(f"\t* Expected: {expected}")
    print(f"\t* Most similar songs by feeling: \n{as_rank(as_indices(most_similar_by_feeling))}")
    print(f"\t* Most similar songs by lyrics: \n{as_rank(as_indices(most_similar_by_lyrics))}")
    print("\n")

Input phrase: I am on a disordered pilgrimage from occult dread toward desperate romance, a numbed figure pushing himself to feel anything at all.
	* Expected: Station to Station
	* Most similar songs by feeling: 
		1. Station To Station
		2. After Today
		3. Sector Z
		4. Saviour Machine
		5. Some Weird Sin
	* Most similar songs by lyrics: 
		1. Golden Years (Instrumental)
		2. Nathan Adler - Segue 1
		3. Sense Of Doubt
		4. Lady Stardust - (Take 1 Alternative Version)
		5. Art Decade


Input phrase: I feel energized and want to dance with my beloved.
	* Expected: Let's Dance
	* Most similar songs by feeling: 
		1. A Better Future
		2. The Passenger
		3. I Wanna Be Your Dog
		4. Bang Bang
		5. New Angels Of Promise
	* Most similar songs by lyrics: 
		1. Land Of 1,000 Dances
		2. Soul Love - (demo)
		3. Baby Grace (A Horrid Cassette)
		4. Let's Dance - (demo)
		5. Stupidity


Input phrase: I'm feeling regretful for falling so low.
	* Expected: Ashes to Ashes
	* Most similar songs by fe

## Transformers

In [44]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

/home/fabio/miniconda3/envs/maua/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1839.22it/s]


In [45]:
feelings_transformer_matrix = model.encode(lyrics_clean['feelings'].to_list())

lyrics_transformer_matrix = model.encode(lyrics_clean['lyrics'].to_list())


In [46]:

test(
    lambda x: model.encode([x]).reshape(1, -1),
    feelings_transformer_matrix,
    lambda x: model.encode([x]).reshape(1, -1),
    lyrics_transformer_matrix
)

Input phrase: I am on a disordered pilgrimage from occult dread toward desperate romance, a numbed figure pushing himself to feel anything at all.
	* Expected: Station to Station
	* Most similar songs by feeling: 
		1. Station To Station
		2. Quicksand - (demo)
		3. The Width Of A Circle
		4. Turn Blue
		5. The Man Who Sold The World
	* Most similar songs by lyrics: 
		1. The Voyeur Of Utter Destruction (As Beauty)
		2. Fill Your Heart
		3. Loving The Alien
		4. Like A Rolling Stone
		5. Quicksand - (demo)


Input phrase: I feel energized and want to dance with my beloved.
	* Expected: Let's Dance
	* Most similar songs by feeling: 
		1. Let's Dance
		2. Land Of 1,000 Dances
		3. Harlem Shuffle
		4. As The World Falls Down
		5. Tiny Girls
	* Most similar songs by lyrics: 
		1. I Feel Free
		2. Let's Dance - (demo)
		3. John, I'm Only Dancing
		4. Cosmic Dancer
		5. Dancing In The Street


Input phrase: I'm feeling regretful for falling so low.
	* Expected: Ashes to Ashes
	* Most similar